# Patient Journey Combine (NBA 3.1)

3.1 copy of `06_patient_journey_combine`: joins the 3.1 lab, Dx and Rx journeys with the 3.1 Px table and the 3.1 diabetes-type / HF flags (PJ04), and writes the same five patient-month tables as 3.0. The 3.0 notebook is not changed (only its `%run` path was fixed after the folder move).

**What changes vs 3.0**
* Inputs: the dated run folder `RUN_OUTPUT_DIR` (all five 3.1 tables), instead of the undated validation tables.
* `DIABETIC_FLAG` comes from `T2D_FLAG_30` of `kerendia_nba_3_1_dm_hf_flags_validation` (same 3.0 rule, one row per patient), instead of the 3.0 T2D table.
* `DIALYSIS_FLAG`: same rule, from the 3.1 Px table (`FIRST_DIALYSIS_DATE`, basket codes).
* New in `pat_sglt2_filter_m`: `HF_FLAG` per month (month >= month of `HF_FIRST_DATE`) and `DM_TYPE`. All 3.0 columns are unchanged, so downstream notebooks still work.
* `cohort_label` from the three journeys is merged into one column (the 3.0 join kept three copies).
* **C11 (2026-11-24), CKD stage for eligibility:** cell 2b builds `CKD_STAGE_31` from actual eGFR / UACR tests and coded Dx stages of the last 12 months (lab first; eGFR 60+ only with UACR >= 30 or an N18 code, else unknown; variant B keeps the SGLT2 rule for unknown stage). Inputs: the PJ01 / PJ02 step-4 pulls. Only the eligibility flags use it; new columns `CKD_STAGE_31` and `CKD_STAGE_31_SRC` in `pat_sglt2_filter_m`. `CKD_STAGE_RULE = "3.0"` restores the old stage. Tested on a 5% sample of `full_2026_11_16`: eligible patient-months -16.4%, eligible patients -17.7%, HF +0.5%.

**Run order:** 01 lab, 02 Dx, 03 Rx, then 05 Px and 04 DM / HF flags (they read the patients of 01-03), then this notebook. Cell 2b also reads the step-4 lab / Dx pulls that a full run of 01 / 02 writes (validation runs read those of `HCP_LATEST_FULL_RUN`).

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../../00_cohort_config"

In [0]:
# 1. Run folder: the five 3.1 inputs are read from here and the five outputs are written here.
#    RUN_OUTPUT_DIR and VALIDATION_MODE come from 00_cohort_config (job parameters; default validation_2026_10_30/).
print(f"VALIDATION_MODE = {VALIDATION_MODE} | RUN_OUTPUT_DIR = {RUN_OUTPUT_DIR}")
INPUTS = {
    "lab":   "kerendia_nba_3_1_lab_journey_validation",     # 01: PATIENT_ID, DATE, GFR_STAGE, UACR_STAGE
    "dx":    "kerendia_nba_3_1_dx_journey_validation",      # 02: DX_STAGE_UPDATED, ...
    "rx":    "kerendia_nba_3_1_rx_journey_validation",      # 03: TREATMENT_CATEGORY, CATEGORY (arrays)
    "px":    "kerendia_nba_3_1_px_validation",              # 05: one row per patient, FIRST_DIALYSIS_DATE
    "flags": "kerendia_nba_3_1_dm_hf_flags_validation",     # 04: one row per patient, T2D_FLAG_30, DM_TYPE, HF_FIRST_DATE
}
dfs = {k: read_delta(RUN_OUTPUT_DIR + t) for k, t in INPUTS.items()}   # Delta folders or UC tables (STORAGE_MODE)
for k, d in dfs.items():
    print(f"{k:6s} {d.count():>9,} rows, {d.select('PATIENT_ID').distinct().count():>6,} patients | {d.columns}")
for k in ("px", "flags"):
    assert dfs[k].count() == dfs[k].select("PATIENT_ID").distinct().count(), f"{k}: more than one row per patient"

In [0]:
# 2. Same join as 3.0; cohort_label from the three journeys merged into one column
_lab = dfs["lab"].withColumnRenamed("cohort_label", "_cl_lab")
_dx = dfs["dx"].withColumnRenamed("cohort_label", "_cl_dx")
_rx = dfs["rx"].withColumnRenamed("cohort_label", "_cl_rx")

df_spine_pat_journey = (
  _lab
  .join(_dx, how="full", on=["PATIENT_ID", "DATE"])
  .join(_rx, how="full", on=["PATIENT_ID", "DATE"])
  .withColumn("cohort_label", F.coalesce("_cl_lab", "_cl_dx", "_cl_rx"))
  .drop("_cl_lab", "_cl_dx", "_cl_rx")
  .withColumn("DX_STAGE_MAX", col("DX_STAGE_UPDATED"))
  .withColumn("GFR_DX_STAGE", F.coalesce(col("DX_STAGE_MAX"), col("GFR_STAGE")))
  .withColumn("GFR_UACR_DX_STAGE", F.coalesce(col("DX_STAGE_MAX"), col("GFR_STAGE"), col("UACR_STAGE")))
)
print(f"Spine: {df_spine_pat_journey.count():,} patient-months, {df_spine_pat_journey.select('PATIENT_ID').distinct().count():,} patients")

In [0]:
# 2b. 3.1 CKD stage for eligibility (C11, decided 2026-11-24: variant B)
# Plan: hcp_level_features/Feature_Evaluation/05_Feature_Views/NBA_3_1_LAAD_Feature_Views_Readiness_Plan.md, sections 6.2-6.3.
# The 3.0 stage of cell 2 (coalesce(DX_STAGE_UPDATED, GFR_STAGE, UACR_STAGE)) carries stale values: DX_STAGE_UPDATED is the
# worst Dx stage ever coded, and the lab journey carries GFR / UACR stage forward with no time limit (PJ01).
# The 3.1 stage uses ACTUAL tests and CODED Dx stages of the last CKD_LOOKBACK_MONTHS months (the month itself included):
#   1. Lab first: stage of the latest eGFR test month (worst test of that month). A Dx stage 5 coded after that test wins.
#   2. eGFR stage 1-2 (60+) counts only with kidney-damage evidence (UACR >= 30 mg/g or an N18 code); otherwise unknown.
#   3. No eGFR test: latest coded Dx stage 1-5.
#   4. Otherwise unknown (null). Variant B: unknown keeps the SGLT2 rule of cell 4 (stage unknown AND ever SGLT2 -> eligible).
# Inputs: the PJ01 / PJ02 step-4 pulls (HCP_LAB31_PULL_TABLE / HCP_DX31_PULL_TABLE, 00_cohort_config). They hold the tests and
# codes of NPI-matched HCPs in the 3.1 HCP universe only. The coded Dx stage equals DX_STAGE_MONTH of the Dx journey
# (checked 2026-11-24 on a 5% patient sample of full_2026_11_16: 3,603,433 / 3,603,433 patient-months).
# Only eligibility (cell 4) uses CKD_STAGE_31; GFR_DX_STAGE / GFR_UACR_DX_STAGE and the other four tables are unchanged.
# CKD_STAGE_RULE = "3.0" brings back the old stage for eligibility (comparison runs).
CKD_STAGE_RULE      = get_run_param("CKD_STAGE_RULE", "3.1")
CKD_LOOKBACK_MONTHS = int(get_run_param("CKD_LOOKBACK_MONTHS", "12"))
assert CKD_STAGE_RULE in ("3.0", "3.1"), f"CKD_STAGE_RULE must be '3.0' or '3.1', got {CKD_STAGE_RULE}"
assert OFFSET_DAY == 0, "The 3.1 CKD stage rule assumes calendar months (OFFSET_DAY = 0)"

def ckd_stage_31(spine, lab_pull, dx_pull, lookback=CKD_LOOKBACK_MONTHS):
    """CKD_STAGE_31 (1-5, null = unknown) and CKD_STAGE_31_SRC (lab / dx / none) per PATIENT_ID x DATE of the spine."""
    pats = spine.select("PATIENT_ID").distinct()
    months = spine.select("PATIENT_ID", col("DATE").alias("M")).distinct().withColumn("_SP", F.lit(1))
    lab = (lab_pull.filter(col("TEST_TYPE").isin("GFR", "UACR")).join(pats, "PATIENT_ID", "left_semi")
           .groupBy("PATIENT_ID", F.trunc("SVC_DT", "month").alias("M"))
           .agg(F.max(F.when(col("TEST_TYPE") == "GFR", col("GFR_STAGE"))).alias("_GFR"),
                F.max(F.when((col("TEST_TYPE") == "UACR") & (col("UACR_STAGE") >= 2), 1)).alias("_UACR30")))
    dx = (dx_pull.join(pats, "PATIENT_ID", "left_semi")
          .groupBy("PATIENT_ID", F.trunc("SVC_DT", "month").alias("M"))
          .agg(F.max(F.when(col("DX_STAGE").between(1, 5), col("DX_STAGE").cast("int"))).alias("_DX"),
               F.max(F.when(col("DIAGNOSIS_CODE").startswith("N18"), 1)).alias("_N18")))
    w = Window.partitionBy("PATIENT_ID").orderBy("M").rowsBetween(Window.unboundedPreceding, 0)
    last = lambda cond, val: F.last(F.when(cond, F.struct(col("M").alias("m"), val.alias("v"))), True).over(w)
    g = (months.join(lab, ["PATIENT_ID", "M"], "full").join(dx, ["PATIENT_ID", "M"], "full")
         .withColumn("_L", last(col("_GFR").isNotNull(), col("_GFR")))
         .withColumn("_U", last(col("_UACR30") == 1, F.lit(1)))
         .withColumn("_N", last(col("_N18") == 1, F.lit(1)))
         .withColumn("_D", last(col("_DX").isNotNull(), col("_DX")))
         .filter(col("_SP") == 1))
    recent = lambda s: col(s).isNotNull() & (F.months_between("M", col(s + ".m")) <= lookback - 1)
    lab_ok, dx_ok, damage = recent("_L"), recent("_D"), recent("_U") | recent("_N")
    gfr, dxs = col("_L.v"), col("_D.v")
    stage = (F.when(lab_ok & dx_ok & (dxs == 5) & (col("_D.m") > col("_L.m")), F.lit(5))
              .when(lab_ok & gfr.isin(3, 4, 5), gfr)
              .when(lab_ok & damage, gfr)
              .when(lab_ok, F.lit(None).cast("int"))   # eGFR 60+ without damage evidence: unknown (variant B)
              .when(dx_ok, dxs))                       # otherwise null = unknown
    src = F.when(lab_ok, "lab").when(dx_ok, "dx").otherwise("none")
    return g.select("PATIENT_ID", col("M").alias("DATE"), stage.alias("CKD_STAGE_31"), src.alias("CKD_STAGE_31_SRC"))

_ckd31 = ckd_stage_31(df_spine_pat_journey,
                      spark.read.format("delta").load(hcp_input_path(HCP_LAB31_PULL_TABLE)),
                      spark.read.format("delta").load(hcp_input_path(HCP_DX31_PULL_TABLE)))
_n_spine = df_spine_pat_journey.count()
df_spine_pat_journey = df_spine_pat_journey.join(_ckd31, on=["PATIENT_ID", "DATE"], how="left")
assert df_spine_pat_journey.count() == _n_spine, "The CKD_STAGE_31 join changed the row count"
ELIG_STAGE_COL = "CKD_STAGE_31" if CKD_STAGE_RULE == "3.1" else "GFR_UACR_DX_STAGE"
print(f"CKD_STAGE_RULE = {CKD_STAGE_RULE}: eligibility uses {ELIG_STAGE_COL} | lookback {CKD_LOOKBACK_MONTHS} months")
print(f"Lab pull: {HCP_LAB31_PULL_TABLE}\nDx pull:  {HCP_DX31_PULL_TABLE}")
display(df_spine_pat_journey.groupBy("CKD_STAGE_31_SRC", "CKD_STAGE_31").count().orderBy("CKD_STAGE_31_SRC", "CKD_STAGE_31"))

In [0]:
# 3a. DIABETIC_FLAG = T2D_FLAG_30 (the 3.0 rule, from PJ04); DM_TYPE and the monthly HF flag are new in 3.1
_flags = dfs["flags"].select(
    "PATIENT_ID",
    col("T2D_FLAG_30").cast("int").alias("DIABETIC_FLAG"),
    "DM_TYPE",
    "HF_TYPE",                                   # C3: HF arm uses HFPEF_PROXY only (LVEF >= 40% proxy)
    F.to_date("HF_FIRST_DATE").alias("HF_FIRST_DATE"))
df_spine_pat_journey_with_flag1 = (
  df_spine_pat_journey
  .join(_flags, how="left", on="PATIENT_ID")
  .na.fill(value=0, subset=["DIABETIC_FLAG"])
  # HF from the month of the first HF code on (same month logic as the dialysis flag)
  .withColumn("HF_FLAG", F.when(F.trunc(col("HF_FIRST_DATE"), "month") <= col("DATE"), 1).otherwise(0))
  .drop("HF_FIRST_DATE")
)

# 3b. DIALYSIS_FLAG: same rule as 3.0, from the 3.1 Px table
df_spine_pat_journey_with_flag2 = (
  df_spine_pat_journey_with_flag1
  .join(dfs["px"].select("PATIENT_ID", "FIRST_DIALYSIS_DATE"), how="left", on="PATIENT_ID")
  .withColumn("DIALYSIS_FLAG",
              F.when(col("FIRST_DIALYSIS_DATE") < '2017-01-01', 1).otherwise(
              F.when(F.trunc(F.to_date(col("FIRST_DIALYSIS_DATE")), "month") <= col("DATE"), 1).otherwise(0)))
  .na.fill(value=0, subset=["DIALYSIS_FLAG"])
  .drop("FIRST_DIALYSIS_DATE")
)

assert df_spine_pat_journey_with_flag2.count() == df_spine_pat_journey.count(), "The flag joins changed the row count"
display(df_spine_pat_journey_with_flag2.agg(
    F.count("*").alias("patient_months"),
    F.sum("DIABETIC_FLAG").alias("months_diabetic"),
    F.sum("HF_FLAG").alias("months_hf"),
    F.sum("DIALYSIS_FLAG").alias("months_dialysis"),
    F.countDistinct(F.when(col("DIABETIC_FLAG") == 1, col("PATIENT_ID"))).alias("patients_diabetic"),
    F.countDistinct(F.when(col("HF_FLAG") == 1, col("PATIENT_ID"))).alias("patients_hf"),
    F.countDistinct(F.when(col("DIALYSIS_FLAG") == 1, col("PATIENT_ID"))).alias("patients_dialysis")))

In [0]:
# 4. The five patient-month tables, same logic as 3.0 (cells copied unchanged, except two new columns in pat_sglt2_filter_m)
_base = df_spine_pat_journey_with_flag2.withColumn("SVC_DT_M", col("DATE"))   # DATE is already at cohort-month grain

pat_new_and_total_treatments = (
  _base
  .withColumn("TOTAL_TREATMENTS", F.greatest(F.size(col("TREATMENT_CATEGORY")), F.lit(0)))
  .withColumn("TOTAL_TREATMENTS_LAST", F.lag("TOTAL_TREATMENTS", 1).over(Window.partitionBy("PATIENT_ID").orderBy("DATE")))
  .withColumn("TOTAL_NEW_TREATMENTS", F.greatest(col("TOTAL_TREATMENTS") - col("TOTAL_TREATMENTS_LAST"), F.lit(0)))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("TOTAL_TREATMENTS").alias("TOTAL_TREATMENTS"),
       F.sum("TOTAL_NEW_TREATMENTS").alias("TOTAL_NEW_TREATMENTS"))
  .withColumn("PULL_DT", F.current_date())
)

pat_new_ckd_stage = (
  _base
  .withColumn("CKD_STAGE", col("GFR_DX_STAGE"))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("CKD_STAGE").alias("CKD_STAGE"))
  .withColumn("CKD_STAGE_LAST", F.lag("CKD_STAGE", 1).over(Window.partitionBy("PATIENT_ID").orderBy("SVC_DT_M")))
  .withColumn("NEW_CKD_STAGE_FLAG", F.when(col("CKD_STAGE") == col("CKD_STAGE_LAST"), 0).otherwise(1))
  .withColumn("NEW_CKD_STAGE", F.when(col("NEW_CKD_STAGE_FLAG") == 1, col("CKD_STAGE")).otherwise(None))
  .select("PATIENT_ID", "SVC_DT_M", "NEW_CKD_STAGE")
  .withColumn("PULL_DT", F.current_date())
)

pat_gfr_dx_stage_and_treatment_category = (
  _base
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
       F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
       F.max("UACR_STAGE").alias("UACR_STAGE"),
       F.collect_set("TREATMENT_CATEGORY").alias("CATEGORY"),
       F.collect_set("CATEGORY").alias("BRAND"))
  .withColumn("PULL_DT", F.current_date())
)

pat_uacr_stage_and_treatment_category = (
  pat_gfr_dx_stage_and_treatment_category.drop("PULL_DT")
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .withColumn("BRAND", F.explode_outer(col("BRAND")))
  .withColumn("CATEGORY", F.regexp_replace("CATEGORY", r"[\s]*$", ""))
  .withColumn("PULL_DT", F.current_date())
)

# SGLT2 patients and eligibility.
# 3.1 rule (decided 2026-11-23, C1 / C9 / C3 / C10), following the Kerendia label: CKD + T2D, CKD + T1D (approved 2026-09-17),
# HF with LVEF >= 40% (approved 2025-07-14).
#   STAGE_ELIGIBLE: null / 0 -> -1 (unknown); 1 or 5 -> 0 (not eligible); 2-4 -> 1.
#       C11 (2026-11-24): the stage is ELIG_STAGE_COL from cell 2b (CKD_STAGE_31 by default: actual tests and coded Dx of the
#       last 12 months, eGFR 60+ only with damage evidence; GFR_UACR_DX_STAGE with CKD_STAGE_RULE = "3.0").
#   ELIGIBLE_CKD (C1 + C9 + C10): no dialysis AND DM_TYPE in (T2D, T1D) AND (stage 2-4 OR (stage unknown AND ever SGLT2)).
#       C1 fix: 3.0 compared GFR_UACR_DX_STAGE == -1, which is never true, so the SGLT2 rule never applied.
#       C10: DM_TYPE = OTHER (only E08 / E09 / E13 codes: secondary, drug-induced, other specified diabetes) is not on label.
#   ELIGIBLE_HF (C3): HF_FLAG = 1 in the month AND HF_TYPE = HFPEF_PROXY AND no dialysis AND stage != 5. No diabetes needed.
#   ELIGIBLE_PATIENT_FLAG = ELIGIBLE_CKD OR ELIGIBLE_HF (same column name as 3.0, so downstream notebooks still work).
pat_sglt2_flag = (
  _base
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .filter("TREATMENT_CATEGORY = 'SGLT2'")
  .withColumn("SGLT2_FLAG", F.lit(1))
  .select("PATIENT_ID", "SGLT2_FLAG").distinct()
)
pat_sglt2_filter = (
  _base
  .withColumn("STAGE_ELIGIBLE",
              F.when((col(ELIG_STAGE_COL).isNull()) | (col(ELIG_STAGE_COL) == 0), -1).otherwise(
              F.when((col(ELIG_STAGE_COL) == 1) | (col(ELIG_STAGE_COL) == 5), 0).otherwise(1)))
  .join(pat_sglt2_flag, how="left", on="PATIENT_ID")
  .na.fill(value=0, subset="SGLT2_FLAG")
  .withColumn("_NO_DIALYSIS", col("DIALYSIS_FLAG") == 0)
  .withColumn("_DM_ON_LABEL", col("DM_TYPE").isin("T2D", "T1D"))
  .withColumn("_CKD_STAGE_OK", (col("STAGE_ELIGIBLE") == 1) | ((col("STAGE_ELIGIBLE") == -1) & (col("SGLT2_FLAG") == 1)))
  .withColumn("ELIGIBLE_CKD_T2D", F.when(col("_NO_DIALYSIS") & col("_CKD_STAGE_OK") & (col("DM_TYPE") == "T2D"), 1).otherwise(0))
  .withColumn("ELIGIBLE_CKD_T1D", F.when(col("_NO_DIALYSIS") & col("_CKD_STAGE_OK") & (col("DM_TYPE") == "T1D"), 1).otherwise(0))
  .withColumn("ELIGIBLE_HF", F.when(col("_NO_DIALYSIS") & (col("HF_FLAG") == 1) & (col("HF_TYPE") == "HFPEF_PROXY")
                                    & ~F.coalesce(col(ELIG_STAGE_COL) == 5, F.lit(False)), 1).otherwise(0))
  .withColumn("ELIGIBLE_PATIENT_FLAG", F.greatest("ELIGIBLE_CKD_T2D", "ELIGIBLE_CKD_T1D", "ELIGIBLE_HF"))
  .drop("_NO_DIALYSIS", "_DM_ON_LABEL", "_CKD_STAGE_OK")
)
_stage = lambda c, top: F.when(col(c) == 2, "STAGE_2").otherwise(F.when(col(c) == 3, "STAGE_3").otherwise(
    F.when(col(c) == 4, "STAGE_4").otherwise("STAGE_UNKNOWN") if top == 4 else F.lit("STAGE_UNKNOWN")))
pat_sglt2_filter_m = (
  pat_sglt2_filter
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("ELIGIBLE_PATIENT_FLAG").alias("ELIGIBLE_PATIENT_FLAG"),
       F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
       F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
       F.max("UACR_STAGE").alias("UACR_STAGE"),
       F.max("HF_FLAG").alias("HF_FLAG"),          # new in 3.1
       F.max("DM_TYPE").alias("DM_TYPE"),          # new in 3.1 (one value per patient)
       F.max("HF_TYPE").alias("HF_TYPE"),          # new in 3.1 (one value per patient)
       F.max("ELIGIBLE_CKD_T2D").alias("ELIGIBLE_CKD_T2D"),   # new in 3.1 (C3)
       F.max("ELIGIBLE_CKD_T1D").alias("ELIGIBLE_CKD_T1D"),   # new in 3.1 (C3)
       F.max("ELIGIBLE_HF").alias("ELIGIBLE_HF"),             # new in 3.1 (C3)
       F.max("CKD_STAGE_31").alias("CKD_STAGE_31"),           # new in 3.1 (C11): stage 1-5, null = unknown
       F.max("CKD_STAGE_31_SRC").alias("CKD_STAGE_31_SRC"))   # new in 3.1 (C11): lab / dx / none
  .withColumn("GFR_UACR_DX_STAGE", _stage("GFR_UACR_DX_STAGE", 4))
  .withColumn("GFR_DX_STAGE", _stage("GFR_DX_STAGE", 4))
  .withColumn("UACR_STAGE", _stage("UACR_STAGE", 3))
  .withColumn("PULL_DT", F.current_date())
)

OUTPUTS = {
    "kerendia_nba_3_1_pat_new_and_total_treatments_validation": pat_new_and_total_treatments,
    "kerendia_nba_3_1_pat_new_ckd_stage_validation": pat_new_ckd_stage,
    "kerendia_nba_3_1_pat_gfr_dx_stage_validation": pat_gfr_dx_stage_and_treatment_category,
    "kerendia_nba_3_1_pat_uacr_stage_validation": pat_uacr_stage_and_treatment_category,
    "kerendia_nba_3_1_pat_sglt2_filter_m_validation": pat_sglt2_filter_m,
}
for k, d in OUTPUTS.items():
    print(f"{k}: {d.columns}")

In [0]:
# 5. Write to the run folder (same table names as 3.0; the undated *_validation tables are not touched)
WRITE_OUTPUT = True
if WRITE_OUTPUT:
    for name, d in OUTPUTS.items():
        _w = write_delta(d, RUN_OUTPUT_DIR + name)   # Delta folder or UC table (STORAGE_MODE), read back
        print(f"Written {name}: {_w.count():,} rows, {_w.select('PATIENT_ID').distinct().count():,} patients")
else:
    print("WRITE_OUTPUT = False: nothing written")

In [0]:
# 6. Compare pat_sglt2_filter_m with the 3.0 combine output (undated table, built from the 3.0 journeys) on the
#    patient-months both have: same / different eligibility and stage, plus months only in one of them.
#    The 3.0 table covers the 10 validation HCPs only, so this check runs in validation mode only.
if not VALIDATION_MODE:
    dbutils.notebook.exit("Full run: written; 3.0 comparison skipped (the 3.0 table covers the validation HCPs only)")
_QO = _QUERY_OUTPUT   # undated query_output folder (00_cohort_config)
_k = ["PATIENT_ID", "SVC_DT_M"]
_old = read_delta(_QO + "kerendia_nba_3_1_pat_sglt2_filter_m_validation") \
    .select(col("PATIENT_ID").cast("string").alias("PATIENT_ID"), F.to_date("SVC_DT_M").alias("SVC_DT_M"),
            col("ELIGIBLE_PATIENT_FLAG").alias("ELIG_30"), col("GFR_UACR_DX_STAGE").alias("STAGE_30"), F.lit(1).alias("IN_30"))
_new = read_delta(RUN_OUTPUT_DIR + "kerendia_nba_3_1_pat_sglt2_filter_m_validation") \
    .select(col("PATIENT_ID").cast("string").alias("PATIENT_ID"), F.to_date("SVC_DT_M").alias("SVC_DT_M"),
            col("ELIGIBLE_PATIENT_FLAG").alias("ELIG_31"), col("GFR_UACR_DX_STAGE").alias("STAGE_31"), F.lit(1).alias("IN_31"))
_j = _old.join(_new, _k, "full")
_both = (col("IN_30") == 1) & (col("IN_31") == 1)
display(_j.agg(
    F.sum(F.when(_both, 1).otherwise(0)).alias("months_in_both"),
    F.sum(F.when(col("IN_31").isNull(), 1).otherwise(0)).alias("months_only_30"),
    F.sum(F.when(col("IN_30").isNull(), 1).otherwise(0)).alias("months_only_31"),
    F.sum(F.when(_both & (col("ELIG_30") == col("ELIG_31")), 1).otherwise(0)).alias("same_eligibility"),
    F.sum(F.when(_both & (col("ELIG_30") == 0) & (col("ELIG_31") == 1), 1).otherwise(0)).alias("elig_0_to_1"),
    F.sum(F.when(_both & (col("ELIG_30") == 1) & (col("ELIG_31") == 0), 1).otherwise(0)).alias("elig_1_to_0"),
    F.sum(F.when(_both & (col("STAGE_30") != col("STAGE_31")), 1).otherwise(0)).alias("stage_changed"),
    F.countDistinct(F.when(col("IN_31") == 1, col("PATIENT_ID"))).alias("patients_31"),
    F.countDistinct(F.when(col("IN_30") == 1, col("PATIENT_ID"))).alias("patients_30")))
# eligible patient-months per month, 3.0 vs 3.1 (from 2025, the cohort window)
display(_j.filter(col("SVC_DT_M") >= "2025-01-01").groupBy("SVC_DT_M").agg(
    F.sum(F.coalesce("ELIG_30", F.lit(0))).alias("eligible_30"),
    F.sum(F.coalesce("ELIG_31", F.lit(0))).alias("eligible_31")).orderBy("SVC_DT_M"))